# Supervised staged retail clip classification

Inspect the MNNIT version 1 data, frozen split, lightweight supervised baseline and errors. This notebook reuses verified cached results; it does not retrain the encoder or replace the RetailS pose baselines. Labels apply to entire staged laboratory clips, not acting people or exact theft times.

In [ ]:
from pathlib import Path
import json, hashlib, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
assert (ROOT / "configs/mnnit_clip_v1.json").is_file()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
OUT = ROOT / "outputs/mnnit_clip_classification"
def digest(path):
    with path.open("rb") as f: return hashlib.file_digest(f, "sha256").hexdigest()
print("Python:", sys.executable)
print("Repository:", ROOT)

## 1 Verify the original data

The archive has 90 normal and 92 positive publisher-labeled clips. Every source frame was decoded by the preparation script. Original FPS is 30; sizes include 640x480 and 1920x1080. Three exact duplicates are excluded.

In [ ]:
data = json.loads((OUT / "data_report.json").read_text())
assert digest(ROOT / "data/action_sources/mnnit_v1/Dataset.zip") == data["archive_sha256"]
assert digest(OUT / "inventory.csv") == data["inventory_sha256"]
assert digest(OUT / "sampled_rgb.npz") == data["rgb_cache_sha256"]
display(pd.DataFrame([{k:v for k,v in data.items() if k in ["videos","canonical_videos","exact_duplicates_excluded","near_duplicate_pairs","decoded_frames","decoded_all_frames"]}]))
inventory = pd.read_csv(OUT / "inventory.csv")
display(inventory.groupby(["class_name", "included"]).size().rename("clips").reset_index())

## 2 Inspect the scene and labels

A sampled midpoint from each original clip is available in the contact sheets. These are laboratory scenes with repeated clothing and backgrounds. Preview inspection does not independently verify every action label.

In [ ]:
for name in ["contact_01.jpg", "contact_06.jpg"]:
    image = cv2.imread(str(OUT / name))
    assert image is not None
    plt.figure(figsize=(13, 12)); plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()

## 3 Check the frozen split

Seed 2026. Exact and detected sampled near-duplicate components remain within one split. The release lacks verified actor/session IDs; this does not demonstrate independence across people or recording sessions.

In [ ]:
from src.data.clip_classification import verify_split, portable_probability
split = json.loads((OUT / "frozen_split.json").read_text())
verify_split(split["rows"])
display(pd.DataFrame(split["counts"]).T.rename(columns={"0":"normal", "1":"positive"}))
config = json.loads((ROOT / "configs/mnnit_clip_v1.json").read_text())
assert digest(ROOT / "configs/mnnit_clip_v1.json") == split["config_sha256"]
display(config)

## 4 Verify models and measured metrics

The primary model uses mean, standard deviation and absolute consecutive changes of 576-dimensional frozen frame embeddings. The control omits changes. The scaler and logistic head use training clips only; C=1, balanced classes and fixed threshold 0.5. No test-based tuning is performed.

In [ ]:
report = json.loads((OUT / "evaluation.json").read_text())
assert report["complete"] and not report["test_selected_parameters"] and not report["old_pose_models_changed"]
assert digest(OUT / "frozen_split.json") == report["frozen_split_sha256"]
assert digest(OUT / "predictions.csv") == report["predictions_sha256"]
for name, expected in report["model_sha256"].items():
    assert digest(OUT / (name + "_model.npz")) == expected
rows = [{"model":name,"split":subset,**values} for name, subsets in report["metrics"].items() for subset, values in subsets.items()]
display(pd.DataFrame(rows)[["model","split","clips","accuracy","balanced_accuracy","precision","recall","f1","roc_auc","tp","tn","fp","fn"]])

## 5 Examine confusion counts and errors

There are 36 test clips. The primary model has 7 false positives and 4 false negatives. Validation accuracy is higher than test accuracy; these results remain provisional. Test-error examples are diagnostic, not a new tuning set.

In [ ]:
for name in ["confusion.png", "test_error_examples.jpg"]:
    image = cv2.imread(str(OUT / name))
    assert image is not None
    plt.figure(figsize=(13, 10)); plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
display(pd.read_csv(OUT / "test_errors.csv"))

## 6 Check portable inference

The NPZ contains the scaler and logistic coefficients. Scores should reproduce the saved predictions without loading a pickle or training a model. Scores are not calibrated theft probabilities.

In [ ]:
with np.load(OUT / "frozen_features.npz", allow_pickle=False) as features:
    matrix = features["features"]; clip_ids = features["clip_ids"].tolist()
with np.load(OUT / "appearance_and_changes_model.npz", allow_pickle=False) as model:
    probability = portable_probability(matrix, model["mean"], model["scale"], model["coefficient"], model["intercept"])
predictions = pd.read_csv(OUT / "predictions.csv")
primary = predictions[predictions.model == report["primary_model"]]
actual = np.array([probability[clip_ids.index(clip)] for clip in primary.clip_id])
np.testing.assert_allclose(actual, primary.positive_score, rtol=1e-7, atol=1e-9)
print("Portable inference matches all", len(primary), "retained clip scores.")
print("Full regression evidence:", OUT / "regression_tests.txt")

## Interpretation and next research gate

This completes a lightweight supervised feasibility experiment. The dataset, supervision and split differ from RetailS, so its metrics are not proof of improvement on the prior benchmark. Actor/session independence, real-store generalization and person/time localization remain unverified. Temporal pooling loses detailed order, and feature changes are not optical flow. Preserve the exposed test split as historical evidence. A stronger temporal model should be evaluated under a fresh, predeclared independent protocol.

Source and attribution: Ansari and Singh, Mendeley Data V1, DOI 10.17632/r3yjf35hzr.1, CC BY 4.0. See MNNIT_CLIP_CLASSIFICATION.md for reproduction and runtime details.